# NB04 — Calibration（教相機同投影儀認識彼此）
三角化需要知道：相機內參 (K, dist)、投影儀內參、同佢哋之間嘅 (R, T)。
做法：影一堆已知幾何嘅標定板（chessboard），cv2 幫手解。
投影儀嘅標定用一個絕妙 trick：**當佢係第二部相機**——播圖案落標定板，
decode 出每個角點對應嘅投影儀列 = 「投影儀睇到嘅影像」。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## 1. 合成標定板（可控實驗）
先用虛擬相機渲染 chessboard 視圖——ground truth 已知，可以驗證標定質素。

In [2]:
import cv2
from sl_edu import calibrate

board = (9, 6)          # inner corners
sq = 25.0               # mm per square
K_true = np.array([[525., 0, 320], [0, 525., 240], [0, 0, 1]])
size = (640, 480)

views = [(np.array([0.1,0.1,0.05]), np.array([0.,0.,600.])),
         (np.array([-0.15,0.1,0.2]), np.array([30.,-20.,650.])),
         (np.array([0.2,-0.1,-0.15]), np.array([-40.,10.,550.])),
         (np.array([0.,0.2,0.1]), np.array([10.,30.,700.]))]

objp = np.zeros((board[0]*board[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:board[0], 0:board[1]].T.reshape(-1, 2) * sq

img_points = []
for rvec, tvec in views:
    pts, _ = cv2.projectPoints(objp, rvec, tvec, K_true, np.zeros(5))
    img_points.append(pts.reshape(-1, 1, 2).astype(np.float32))
print(f'{len(img_points)} synthetic views rendered')

4 synthetic views rendered


## 2. calibrateCamera：由多視圖解內參

In [3]:
cal = calibrate.calibrate_camera(objp, img_points, size)
print(f"RMS reprojection error: {cal['rms']:.4f} px")
print(f"fx: true 525.0, recovered {cal['K'][0,0]:.2f}")
print(f"cx: true 320.0, recovered {cal['K'][0,2]:.2f}")

RMS reprojection error: 0.0000 px
fx: true 525.0, recovered 525.00
cx: true 320.0, recovered 320.00


## 3. 真實世界嘅標定結果：`caliInfo.yml`
repo 入面有呢台真實 rig（1280×1024 相機 + 投影儀）嘅完整標定。
生產系統用**同心環標定板**（中心定位比 chessboard 角點更穩）——
原理一樣，角點檢測換咗做 ring centroid（見 `src/calibration/`）。

In [4]:
fs = cv2.FileStorage(str(DATA / 'monocularCamera' / 'caliInfo.yml'),
                     cv2.FILE_STORAGE_READ)
calib = {k: fs.getNode(k).mat() for k in ('M1', 'D1', 'M4', 'D4', 'Rlp', 'Tlp')}
fs.release()

print('camera K (M1):'); print(np.round(calib['M1'], 1))
print('projector K (M4):'); print(np.round(calib['M4'], 1))
print(f"baseline |T| = {np.linalg.norm(calib['Tlp']):.1f} mm")
print(f"camera resolution: {DATA/'monocularCamera'/'L'} has",
      len(list((DATA/'monocularCamera'/'L').glob('*.bmp'))), 'calibration views')

camera K (M1):
[[1.7541e+03 0.0000e+00 6.4540e+02]
 [0.0000e+00 1.7529e+03 5.2950e+02]
 [0.0000e+00 0.0000e+00 1.0000e+00]]
projector K (M4):
[[2.4966e+03 0.0000e+00 9.9030e+02]
 [0.0000e+00 2.4958e+03 5.4700e+02]
 [0.0000e+00 0.0000e+00 1.0000e+00]]
baseline |T| = 109.4 mm
camera resolution: /private/tmp/slmaster-fork/data/monocularCamera/L has 18 calibration views


留意 T 嘅模 ≈ 108mm——就係相機同投影儀嘅**基線距離**。
基線越長深度越準，但遮擋越多；呢個係硬件設計嘅 fundamental trade-off。

下一課：齊料！decode + 標定 → 真。三角化 → 3D 點雲。